# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати памʼяті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

In [1]:
# trl — бібліотека для донавчання мовних моделей (SFT, DPO тощо),
# peft — реалізація LoRA та інших методів PEFT, bitsandbytes — 4-бітна квантизація для QLoRA
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "bitsandbytes" "datasets" "accelerate"
# Colab має застарілу бібліотеку torchao, з якою peft не імпортується; для цього демо вона не потрібна
!pip uninstall -q -y torchao

In [2]:
import gc
import json
import os
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Для цього ноутбука потрібна GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) —
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` — відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` — конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [3]:
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}


In [4]:
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [5]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` — коротка інструкція;
- `user` — повідомлення клієнта;
- `assistant` — правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [6]:
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

### Тренувальний, валідаційний і тестовий набори

- **Тренувальний** (2800 прикладів) — на ньому модель вчиться.
- **Валідаційний** (200) — на ньому під час тренування обчислюється validation loss, щоб помітити перенавчання.
- **Тестовий** (1000) — на ньому лише один раз, після тренування, рахуємо підсумкові метрики.
  Під час тренування й налаштування ці приклади не використовуються, тому оцінка на них найчесніша.

Після дедуплікації та перемішування тестовий набір беремо з прикладів, які не потрапили до перших 3000.

Тестовий набір більший за валідаційний, бо від його розміру залежить точність вимірювання метрик:
на 1000 прикладах одна помилка змінює точність на 0,1%, тоді як на 200 — на 0,5%.
Донавчені моделі помиляються рідко, і на малому тестовому наборі різниця між ними може бути непомітною.

In [7]:
shuffled = dedup_dataset.shuffle(seed=42)

dataset = shuffled.select(range(3000)).map(to_messages, remove_columns=shuffled.column_names)
dataset = dataset.train_test_split(test_size=200, seed=42)
train_dataset, val_dataset = dataset["train"], dataset["test"]
test_dataset = shuffled.select(range(3000, 4000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
train_dataset[0]["messages"]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'content': 'Classify the customer message. Reply with JSON only.',
  'role': 'system'},
 {'content': 'I have got to locate hte bills from {{Person Name}}',
  'role': 'user'},
 {'content': '{"category": "INVOICE", "intent": "check_invoice"}',
  'role': 'assistant'}]

## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сімʼї з однаковою архітектурою й даними попереднього тренування — **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [8]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки «до» передаємо його в промпті разом з описом формату.

In [9]:
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [10]:
def classify_batch(model, messages_batch, system_prompt, max_new_tokens=60):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    # Для генерації батчем короткі промпти доповнюються зліва, щоб відповіді всіх промптів починалися в одній позиції
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар «звернення — відповідь моделі»."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє памʼять GPU після видалення моделей."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` — ранг матриць адаптера: що більший, то більше параметрів і «свободи» має адаптер;
- `lora_alpha` — масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` — dropout в адаптері для регуляризації;
- `target_modules="all-linear"` — адаптери додаються до всіх лінійних шарів (attention і MLP).

In [11]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

Параметри тренування (`SFTConfig`):
- `assistant_only_loss=True` — loss рахується лише на відповідях асистента, а не на репліках користувача й системному промпті:
  для решти токенів мітки замінюються значенням −100, і функція втрат їх пропускає;
- `max_steps=100` — кількість кроків тренування; за один крок модель бачить 8 прикладів, тобто за все тренування — 800 із 2800,
  менше третини епохи. Для вузької задачі цього достатньо: validation loss виходить на плато раніше;
- `learning_rate=2e-4` — для LoRA learning rate приблизно в 10 разів більший, ніж для повного донавчання;
- `eval_strategy="steps"` і `eval_steps=10` — кожні 10 кроків модель оцінюється на валідаційному наборі:
  якщо validation loss почне зростати, модель перенавчається;
- `fp16=True` — змішана точність (T4 не підтримує bf16);
- `gradient_checkpointing=True` — активації не зберігаються, а перераховуються під час зворотного проходу: повільніше, але потрібно менше памʼяті.

In [12]:
def make_config(output_dir, learning_rate=2e-4):
    return SFTConfig(
        output_dir=output_dir,
        max_length=256,                  # діалог із цього датасету після шаблону займає близько 100 токенів
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # ефективний розмір батча: 4 × 2 = 8 прикладів
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        max_steps=100,
        eval_strategy="steps",
        eval_steps=10,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )


def train(model, output_dir, peft_config=peft_config, learning_rate=2e-4):
    """Тренує модель (з LoRA, якщо задано peft_config) і повертає тренер, тривалість, пікову памʼять GPU та validation loss."""
    torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(
        model=model,
        args=make_config(output_dir, learning_rate),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    # Для QLoRA бібліотека TRL переводить ваги адаптера в bf16, який T4 не підтримує.
    # Повертаємо їх у fp32: змішана точність fp16 потребує, щоб параметри, що тренуються, були у fp32
    if getattr(model, "is_loaded_in_4bit", False):
        for param in trainer.model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
    if peft_config is not None:
        trainer.model.print_trainable_parameters()
    start = time.time()
    trainer.train()
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    stats = {
        "Тривалість, хв": (time.time() - start) / 60,
        "Пікова памʼять GPU, GB": torch.cuda.max_memory_allocated() / 1e9,
        "Validation loss": eval_losses[-1],
    }
    return trainer, stats

## 5. LoRA на моделях різного розміру

Для кожної моделі виконуємо однакові кроки:
1. завантажуємо модель у fp16 (ваги Qwen3 збережено в bfloat16, але T4 не підтримує його апаратно);
2. оцінюємо її **до донавчання** з детальним промптом, у якому перелічено всі мітки;
3. донавчаємо методом LoRA з однаковими параметрами;
4. оцінюємо **після донавчання** з коротким промптом без переліку міток;
5. зберігаємо адаптер і вивантажуємо модель з GPU, щоб звільнити памʼять для наступної.

In [13]:
results = {}


def run_lora(model_name):
    """Повний цикл для однієї моделі: оцінка до, LoRA, оцінка після, збереження адаптера."""
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    print(f"{short_name}: {model.num_parameters() / 1e9:.2f} млрд параметрів, "
          f"памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("\nДо донавчання:")
    show(before_answers, k=3)

    trainer, stats = train(model, f"{short_name}-lora")
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("\nПісля донавчання:")
    show(after_answers, k=3)

    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1e6

    results[short_name] = {"до": before_metrics, "LoRA": after_metrics | stats}
    del trainer, model
    free_memory()
    return pd.DataFrame(results[short_name]).round(2)

In [14]:
run_lora("Qwen/Qwen3-0.6B")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-0.6B: 0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB

До донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (7 outputs left out)

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.362599,0.477965,0.437531,0.894589,4323.000000
20,0.321631,0.142814,0.188090,0.956574,8672.000000
30,0.081151,0.061308,0.072774,0.985931,13032.000000
40,0.059347,0.026849,0.044968,0.991250,17377.000000
50,0.024733,0.011753,0.029027,0.996557,21714.000000
60,0.006357,0.007997,0.018860,0.998110,26070.000000
70,0.011551,0.006565,0.016641,0.998124,30452.000000
80,0.007536,0.005510,0.013986,0.998745,34813.000000
90,0.005160,0.005725,0.013832,0.998755,39116.000000
100,0.015992,0.005622,0.013682,0.998755,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,97.70
"Тривалість, хв",NaN,2.44
"Пікова памʼять GPU, GB",NaN,1.42
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


In [15]:
run_lora("Qwen/Qwen3-1.7B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.993085,0.438007,0.368060,0.905950,4323.000000
20,0.315434,0.133403,0.192618,0.955980,8672.000000
30,0.079961,0.045415,0.050101,0.988109,13032.000000
40,0.050064,0.024857,0.027940,0.989694,17377.000000
50,0.017703,0.017336,0.025229,0.993765,21714.000000
60,0.012273,0.005689,0.014858,0.998740,26070.000000
70,0.012604,0.005700,0.011713,0.996879,30452.000000
80,0.012267,0.005200,0.010645,0.997494,34813.000000
90,0.013403,0.004239,0.010560,0.998122,39116.000000
100,0.009917,0.004022,0.010507,0.998740,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",62.7,99.30
"Точність intent, %",57.4,96.80
"Тривалість, хв",NaN,2.45
"Пікова памʼять GPU, GB",NaN,3.83
Validation loss,NaN,0.00
"Адаптер, MB",NaN,69.78


In [16]:
run_lora("Qwen/Qwen3-4B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.768620,0.299742,0.306302,0.923112,4323.000000
20,0.202398,0.100327,0.100009,0.962126,8672.000000
30,0.065391,0.029637,0.035306,0.989987,13032.000000
40,0.035141,0.018402,0.026427,0.995031,17377.000000
50,0.014093,0.007041,0.015575,0.998422,21714.000000
60,0.008938,0.003766,0.009406,0.998437,26070.000000
70,0.007457,0.005259,0.009024,0.997804,30452.000000
80,0.012313,0.004768,0.008272,0.998110,34813.000000
90,0.015740,0.003368,0.008172,0.998742,39116.000000
100,0.012113,0.003258,0.008074,0.998742,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",88.1,100.00
"Точність intent, %",72.5,97.90
"Тривалість, хв",NaN,3.73
"Пікова памʼять GPU, GB",NaN,8.75
Validation loss,NaN,0.00
"Адаптер, MB",NaN,132.19


### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, памʼять і розмір адаптера.

In [17]:
pd.DataFrame({
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00
"Точність intent, %",25.6,97.70,57.4,96.80,72.5,97.90
"Тривалість, хв",NaN,2.44,NaN,2.45,NaN,3.73
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19


## 6. Повне донавчання Qwen3-0.6B

Модель на 0,6 млрд параметрів потребує для повного донавчання близько 10 GB, тому на T4 її можна донавчити повністю й порівняти з LoRA.

Відмінності від LoRA:
- `peft_config=None` — тренуються всі ваги моделі;
- модель завантажуємо у **fp32**: змішана точність fp16 тримає головну копію ваг у fp32, а в fp16 лише обчислює прохід;
- `learning_rate=2e-5` — для повного донавчання learning rate приблизно в 10 разів менший, ніж для LoRA.

In [18]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map="auto")
trainer, full_stats = train(model, "Qwen3-0.6B-full", peft_config=None, learning_rate=2e-5)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)

del trainer, model
free_memory()

pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"], "Qwen3-0.6B повне": full_metrics | full_stats}).round(2)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.255744,0.288718,0.251618,0.934379,4323.000000
20,0.176103,0.060587,0.091798,0.985634,8672.000000
30,0.049239,0.038266,0.028003,0.990942,13032.000000
40,0.046789,0.025577,0.022165,0.994027,17377.000000
50,0.026981,0.007719,0.010975,0.998120,21714.000000
60,0.002136,0.004413,0.004073,0.998442,26070.000000
70,0.005817,0.004062,0.003158,0.999375,30452.000000
80,0.012955,0.003829,0.004107,0.998437,34813.000000
90,0.009278,0.003025,0.003515,0.999370,39116.000000
100,0.014485,0.002765,0.003435,0.999370,43449.000000


📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,Qwen3-0.6B LoRA,Qwen3-0.6B повне
"Чистий JSON, %",100.00,100.00
"Валідний JSON, %",100.00,100.00
"Точність category, %",99.70,99.50
"Точність intent, %",97.70,98.30
"Тривалість, хв",2.44,1.86
"Пікова памʼять GPU, GB",1.42,10.80
Validation loss,0.01,0.00
"Адаптер, MB",40.42,NaN


## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [19]:
adapter_dir = "Qwen3-4B-lora-adapter"
for name in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, name))
    print(f"{name:<35} {size / 1e6:>8.1f} MB")
print("\nДля порівняння, ваги Qwen3-4B у fp16: ~8 GB")

README.md                                0.0 MB
adapter_config.json                      0.0 MB
adapter_model.safetensors              132.2 MB
chat_template.jinja                      0.0 MB
tokenizer.json                          11.4 MB
tokenizer_config.json                    0.0 MB
training_args.bin                        0.0 MB

Для порівняння, ваги Qwen3-4B у fp16: ~8 GB


Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [20]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
print(type(model).__name__)

# Злиту модель можна зберегти як звичайну: model.save_pretrained("qwen3-4b-merged")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM


Перевіримо злиту модель на власних повідомленнях:

In [21]:
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

📩 Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?
🤖 {"category": "DELIVERY", "intent": "delivery_period"}

📩 I was charged twice for the same order, please help
🤖 {"category": "ORDER", "intent": "cancel_order"}

📩 How can I talk to a real person?
🤖 {"category": "CONTACT", "intent": "contact_human_agent"}

📩 I guess, you have shipped to a wrong address
🤖 {"category": "SHIPPING", "intent": "shipping_error"}



In [22]:
del model
free_memory()

## 8. QLoRA

QLoRA завантажує заморожені ваги Qwen3-4B в **4 бітах** (формат NF4), а адаптери LoRA тренує у звичайній точності.
Код тренування той самий, змінюється лише завантаження моделі.

### Формат NF4

У 4 бітах вага може набувати лише одного з 16 значень. Ваги ділять на блоки по 64, і для кожного блоку
зберігають масштаб — найбільше за модулем значення в блоці. Формати відрізняються тим, як розташовані ці 16 значень:
- **INT4** — 16 рівновіддалених значень. Ваги нейромережі розподілені приблизно нормально: більшість близька до нуля,
  а великих за модулем мало. Тому крайні значення використовуються рідко, а біля нуля їх бракує;
- **NF4 (NormalFloat4)** — 16 значень розташовані за квантилями нормального розподілу: густо біля нуля й рідко на краях.
  Кожне значення використовується приблизно однаково часто, тому помилка квантизації менша, ніж у INT4.

NF4 — формат лише для зберігання: перед кожним множенням ваги деквантизуються у fp16 (`bnb_4bit_compute_dtype`).
Детальніше — у блозі Hugging Face [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)
і в статті [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314).

In [23]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-бітний формат, розрахований на нормально розподілені ваги
    bnb_4bit_compute_dtype=torch.float16,   # обчислення виконуються у fp16
    bnb_4bit_use_double_quant=True,         # квантизуються також масштаби квантизації: ще трохи менше памʼяті
)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", quantization_config=bnb_config, device_map="auto")
print(f"Памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Памʼять GPU після завантаження: 2.7 GB


In [24]:
trainer, qlora_stats = train(model, "Qwen3-4B-qlora")
qlora_metrics, _ = evaluate(trainer.model, SHORT_PROMPT)
qlora_stats

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.482020,0.315657,0.297817,0.923437,4323.000000
20,0.196195,0.127579,0.097366,0.957768,8672.000000
30,0.071569,0.025978,0.042648,0.993443,13032.000000
40,0.029103,0.014845,0.022632,0.996249,17377.000000
50,0.015964,0.012617,0.016194,0.995899,21714.000000
60,0.008319,0.005418,0.008374,0.998728,26070.000000
70,0.006574,0.006251,0.007883,0.998110,30452.000000
80,0.003349,0.006355,0.007162,0.997485,34813.000000
90,0.013278,0.005073,0.007235,0.998422,39116.000000
100,0.011237,0.004870,0.007263,0.998422,43449.000000


{'Тривалість, хв': 5.164064117272695,
 'Пікова памʼять GPU, GB': 3.665665536,
 'Validation loss': 0.004870124626904726}

## 9. Підсумкове порівняння

In [25]:
summary = {
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary["Qwen3-4B QLoRA"] = qlora_metrics | qlora_stats
pd.DataFrame(summary).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA,Qwen3-0.6B повне,Qwen3-4B QLoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00,99.50,99.90
"Точність intent, %",25.6,97.70,57.4,96.80,72.5,97.90,98.30,97.70
"Тривалість, хв",NaN,2.44,NaN,2.45,NaN,3.73,1.86,5.16
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75,10.80,3.67
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00,0.00,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19,NaN,NaN


**Що варто помітити:**
- LoRA тренує близько 1–2% параметрів моделі, а адаптер займає десятки або сотні мегабайт замість гігабайтів для всієї моделі.
- Порівняйте моделі різного розміру: наскільки менша модель після донавчання поступається більшій і скільки вона заощаджує часу й памʼяті.
  Для вузької задачі з тисячами прикладів донавчена мала модель може бути вигіднішою за велику.
- Порівняйте повне донавчання Qwen3-0.6B з LoRA на тій самій моделі за якістю, часом і памʼяттю.
- До донавчання моделі отримують детальний промпт із переліком міток, після — лише короткий:
  формат і значення міток тепер зберігаються у вагах, а не в промпті. Коротший промпт означає швидшу й дешевшу обробку кожного запиту.
- QLoRA потребує помітно менше памʼяті, ніж LoRA на тій самій моделі, але тренується повільніше, бо ваги щоразу деквантизуються під час обчислень.

---
# Відповіді на теоретичні питання 0.1 (питання 22–30)

### 22. Як довго виконувалися клітинки ноутбука 03_sft_lora? Чи були проблеми?

Ноутбук запускався в Google Colab на GPU **Tesla T4 (15.6 GB)**.

| Етап | Час клітинки | з них тренування |
|---|---|---|
| Встановлення бібліотек | 8 с | — |
| Імпорти | 50 с | — |
| Завантаження й підготовка даних | ≈ 10 с | — |
| **Qwen3-0.6B: оцінка до → LoRA → оцінка після** | **≈ 4 хв** | 2.44 хв |
| **Qwen3-1.7B: те саме** | **≈ 5 хв** | 2.45 хв |
| **Qwen3-4B: те саме** | **≈ 10 хв** | 3.73 хв |
| Повне донавчання Qwen3-0.6B | ≈ 2 хв | 1.86 хв |
| Завантаження адаптера й злиття з моделлю | 31 с | — |
| Перевірка на власних повідомленнях | 3 с | — |
| Завантаження Qwen3-4B у 4 бітах | 29 с | — |
| **QLoRA Qwen3-4B + оцінка** | **≈ 7 хв** | 5.16 хв |

Разом ≈ 35 хвилин. Значну частину часу займало не тренування, а завантаження моделей (Qwen3-4B — 8 GB) і оцінка на 1000 тестових прикладах до й після тренування.

**Проблем із запуском не було.** Попередження:
- `unauthenticated requests to the HF Hub` — завантаження без токена Hugging Face;
- `The PEFT config's base_model_name_or_path was renamed from 'Qwen/Qwen3-0.6B' to 'Qwen/Qwen3-1.7B'` — той самий об'єкт `peft_config` використовується для кількох моделей, і бібліотека попереджає, що в ньому змінилася назва базової моделі. На результат не впливає.

### 23. Коментар до виводу «Пам'ять GPU після завантаження». Чи очікуваний результат?

| Модель | Параметрів | Формат | За формулою (параметри × байти) | Виміряно |
|---|---|---|---|---|
| Qwen3-0.6B | 0.60 млрд | fp16, 2 байти | 0.60 × 2 ≈ 1.2 GB | **1.2 GB** |
| Qwen3-1.7B | 1.72 млрд | fp16, 2 байти | 1.72 × 2 ≈ 3.4 GB | **3.5 GB** |
| Qwen3-4B | 4.02 млрд | fp16, 2 байти | 4.02 × 2 ≈ 8.0 GB | **8.1 GB** |
| Qwen3-4B (QLoRA) | 4.02 млрд | NF4, 0.5 байта | ≈ 2.0 GB + 16-бітні ембединги ≈ 2.6–2.8 GB | **2.7 GB** |

**Так, результат очікуваний.** Пам'ять після завантаження майже точно дорівнює «кількість параметрів × 2 байти» для fp16, як ми рахували в лабораторній 4. Невелика різниця (до 0.1 GB) — службові буфери PyTorch.

Для QLoRA пам'ять не рівно вчетверо менша (8.1 → 2.7, тобто в 3 рази), бо квантизуються лише лінійні шари. Таблиця ембедингів (151 936 × 2560 ≈ 0.39 млрд параметрів ≈ 0.78 GB) і нормалізації лишаються в 16 бітах, а для кожного блоку з 64 ваг зберігається ще й масштаб.

Під час **тренування** пам'яті потрібно більше (пікова пам'ять: 1.42 / 3.83 / 8.75 GB для LoRA), бо додаються адаптери, їхні градієнти, стани оптимізатора й активації.

### 24. Що робить рядок with torch.no_grad() у функції classify()?

`torch.no_grad()` **вимикає відстеження градієнтів** для всіх обчислень усередині блоку.

Під час тренування PyTorch запам'ятовує всі проміжні результати (активації) кожного шару, щоб потім порахувати градієнти зворотним проходом (backward). У `classify()` ми лише **використовуємо** модель для передбачення (інференс), нічого не тренуємо, тож градієнти не потрібні. Без `no_grad()`:
- PyTorch зберігав би активації всіх шарів для кожного згенерованого токена, і **пам'ять GPU** витрачалася б даремно. При генерації батчами по 32 приклади для моделі на 4 млрд параметрів це могло б призвести до нестачі пам'яті;
- обчислення були б **повільнішими**.

Тобто `no_grad()` робить інференс швидшим і економнішим за пам'яттю, а результат лишається тим самим.

### 25. Для чого у classify() задано do_sample=False?

`do_sample=False` вмикає **жадібне декодування** (greedy decoding): на кожному кроці модель бере **найімовірніший** токен, а не випадковий згідно з імовірностями.

Для задачі класифікації це потрібно, бо:
1. **Детермінованість:** те саме звернення завжди отримує ту саму відповідь. Система маршрутизації звернень не може видавати різні відділи для одного й того самого повідомлення.
2. **Чесне вимірювання метрик:** порівнюючи моделі до і після донавчання, ми хочемо бачити різницю між моделями, а не випадковий шум. З вибіркою точність змінювалася б від запуску до запуску.
3. **Правильний формат:** при вибірці модель може обрати малоймовірний токен і зіпсувати JSON або «вигадати» мітку, якої немає в переліку. Ми ж хочемо найімовірнішу, тобто найправильнішу з погляду моделі, відповідь.

Творчість і різноманітність (для чого існує вибірка з temperature) тут не потрібні.

### 26. Чому в демо 3 не можна застосувати такий же підхід (заміну голови), як у демо 1?

1. **Інша архітектура моделі.** BERT у демо 1 — encoder: кожен токен «бачить» увесь текст, і токен `[CLS]` на початку збирає інформацію про всю послідовність, тож до нього природно приєднати голову. Qwen3 — **decoder-only** модель з **причинною маскою**: кожен токен бачить лише попередні. Токена `[CLS]` немає, а перший токен узагалі не бачить решти тексту. Голову довелося б ставити на **останній** токен, і модель працювала б у режимі, для якого її не тренували.
2. **Структурована відповідь з двома мітками.** Потрібно визначити і `category` (11 класів), і `intent` (27 класів) та віддати їх у форматі JSON. З головою знадобилися б дві окремі голови й окремий код для формування JSON. LLM же **генерує відповідь текстом** у потрібному форматі одразу.
3. **Модель лишається мовною моделлю.** При SFT ми не змінюємо архітектуру: модель і далі генерує текст звичайною `lm_head`. Тому її можна оцінити **до донавчання** (з детальним промптом вона вже частково вміє задачу: точність intent 25–72%) і використовувати й для інших задач. Нова голова була б ініціалізована випадково і без тренування не вміла б нічого.
4. **Гнучкість:** нову мітку чи нове поле в JSON можна додати, змінивши дані, без зміни архітектури. З LoRA адаптери легко міняти для різних задач на одній базовій моделі.

Технічно для Qwen3 існує й `AutoModelForSequenceClassification` (голова на останньому токені), тобто підхід можна реалізувати. Але для задачі з JSON і кількома полями SFT природніший і зберігає вміння моделі генерувати текст.

### 27. Кількість токенів короткого й детального промпту. Як це вплине на вартість обробки 10 000 звернень на день?

- Короткий промпт (після донавчання): **11 токенів**.
- Детальний промпт (до донавчання, з переліком усіх 11 категорій і 27 намірів): **166 токенів**, тобто в **15 разів** більше.

Різниця **155 токенів на кожен запит**. При 10 000 звернень на день:
- **10 000 × 155 = 1 550 000 зайвих вхідних токенів на день**, ≈ 46.5 млн на місяць;
- з урахуванням самого звернення клієнта (~15–25 токенів) і токенів шаблону запит з детальним промптом займає приблизно **4–5 разів більше** токенів, ніж з коротким.

**Вплив на вартість:**
- при використанні **платного API** вартість вхідних токенів зростає пропорційно, тобто в кілька разів;
- при **власному сервері** більше токенів означає більше часу GPU на обробку кожного запиту (обробка промпту, увага між токенами) і довший час відповіді. Для тієї ж кількості звернень потрібно більше обладнання.

Донавчання дозволяє «перенести» перелік міток і формат з промпту у ваги моделі. Плата за це — одноразове тренування (кілька хвилин на T4), а виграш у швидкості й вартості — на кожному з тисяч щоденних запитів.

### 28. Для чого в датасеті колонка chat_template_kwargs з enable_thinking=False? Що сталося б, якби донавчали в режимі /think?

`chat_template_kwargs` — параметри, які `SFTTrainer` передає в `apply_chat_template` для кожного прикладу. `enable_thinking=False` означає, що тренувальні приклади форматуються в **режимі без міркувань**: шаблон Qwen3 вставляє перед відповіддю асистента **порожній блок `<think>` `</think>`**, а одразу після нього йде JSON.

**Навіщо:** під час використання (`classify_batch`) модель теж викликається з `enable_thinking=False`. Формат тренування й використання мають збігатися (питання 19): модель вчиться, що після порожнього блоку міркувань одразу йде JSON. Нам потрібна швидка й коротка відповідь без міркувань (`max_new_tokens=60`).

**Якби донавчали в режимі /think** (на тих самих прикладах, де відповідь — лише JSON без міркувань):
- модель вчилася б, що в режимі міркування **відповідати треба одразу, без міркувань**. Це «зламало» б її вміння міркувати в режимі `/think` (той самий ефект, що описаний у демо 2 для GSM8K);
- формат тренування не збігався б з форматом використання: під час інференсу модель бачила б порожній блок `<think>`, якого не бачила в тренуванні, і точність могла б упасти;
- або, навпаки, якби ми використовували модель у режимі `/think`, вона могла б почати генерувати довгі міркування, які не вміщуються в `max_new_tokens=60`, і JSON узагалі не з'явився б (як у демо 2, де за 300 токенів модель не встигла дати відповідь).

### 29. Що означає assistant_only_loss=True? На яких токенах модель вчилася б без нього і чим це погано?

`assistant_only_loss=True` означає, що **функція втрат (loss) рахується лише на токенах відповіді асистента**, тобто на JSON `{"category": …, "intent": …}`. Для всіх інших токенів (системний промпт, повідомлення клієнта, службові токени шаблону) мітки замінюються значенням **−100**, і функція втрат їх пропускає. Модель їх «читає», але не вчиться їх передбачати.

**Без цього параметра** модель вчилася б передбачати **всі токени діалогу**: системний промпт, звернення клієнта і JSON.

**Чим це погано для нашої задачі:**
1. **Модель вчилася б не того.** Більшість токенів у прикладі — звернення клієнта й шаблон, а JSON — лише ~15–20 токенів. Значна частина «зусиль» тренування пішла б на те, щоб навчитися **писати повідомлення клієнтів**, а не класифікувати їх.
2. **Системний промпт однаковий у всіх 2800 прикладах.** Модель швидко його «завчила» б, і loss знизився б, хоча якість класифікації від цього не покращилася б. Значення loss вводили б в оману.
3. **Ризик небажаної поведінки:** модель могла б почати продовжувати текст «за клієнта» замість відповіді.
4. **Менше корисного сигналу:** з тим самим числом кроків (100) модель гірше засвоїла б саме те, що потрібно: зв'язок «звернення → мітки».

### 30. Порівняння LoRA і QLoRA (Qwen3-4B) за пам'яттю, часом і якістю. Коли обрати QLoRA?

| | LoRA (fp16) | QLoRA (NF4, 4 біти) |
|---|---|---|
| Пам'ять після завантаження моделі | 8.1 GB | **2.7 GB** (у 3 рази менше) |
| **Пікова пам'ять GPU під час тренування** | 8.75 GB | **3.67 GB** (у 2.4 раза менше) |
| **Тривалість тренування (100 кроків)** | **3.73 хв** | 5.16 хв (на 38% довше) |
| Параметрів, що тренуються | 33.0 млн (0.81%) | 33.0 млн (0.81%) |
| Validation loss | 0.0033 | 0.0049 |
| Чистий / валідний JSON, % | 100 / 100 | 100 / 100 |
| **Точність category, %** | 100.0 | **99.9** |
| **Точність intent, %** | 97.9 | **97.7** |

**Пам'ять:** QLoRA зберігає заморожені ваги в 4 бітах замість 16, тож потребує значно менше пам'яті. Модель на 4 млрд параметрів тренується, займаючи лише 3.7 GB. Це навіть менше, ніж LoRA для Qwen3-1.7B (3.83 GB).

**Час:** QLoRA повільніша на 38%, бо перед кожним множенням ваги **деквантизуються** з NF4 у fp16, і на це йде додатковий час на кожному кроці.

**Якість:** **практично однакова.** Різниця 0.1% в category (1 приклад з 1000) і 0.2% в intent (2 приклади з 1000), тобто в межах випадкових коливань. Validation loss трохи вищий (0.0049 проти 0.0033), бо 4-бітні ваги — наближення оригінальних, але на точність класифікації це майже не вплинуло. Адаптери LoRA тренуються у звичайній точності й «компенсують» похибку квантизації. QLoRA на 4B (97.7% intent) також точніша за LoRA на 1.7B (96.8%), займаючи менше пам'яті.

**Коли я обрала б QLoRA:**
- коли модель **не вміщується в пам'ять GPU** у fp16. Наприклад, модель на 7–8 млрд параметрів у fp16 займає 14–16 GB і не вміщується в T4 з її 15 GB, а з QLoRA займає ~5–6 GB і тренується;
- коли є лише **безкоштовний чи слабкий GPU** (Colab T4, ноутбук з 6–8 GB відеопам'яті), а хочеться взяти більшу, якіснішу модель. Як видно з таблиці, більша модель з QLoRA може бути кращою за меншу модель зі звичайною LoRA;
- коли треба тренувати **кілька моделей одночасно** на одному GPU.

**Коли краще звичайна LoRA:** коли пам'яті вистачає (як для Qwen3-4B на T4): вона швидша, а виграш QLoRA в пам'яті тоді не потрібен.